kernel : Python (Pixi)

#### Standardizing Heterogeneous gene expression Data Formats

Public datasets provided through GEO (Gene Expression Omnibus) and SCA (Single Cell Atlas) are uploaded by individual researchers, each using data structures and file formats that are most convenient for their own experimental workflows. 
As a result, gene expression data lack any consistent or unified format across studies. 
If such heterogeneous data are used directly without prior standardization, it can lead to serious downstream problems, including parsing errors, incorrect sample-to-metadata mappings, incompatible matrix structures, and irreproducible analysis results. 
This standardization step ensures structural consistency, improves pipeline robustness, and minimizes technical variability unrelated to the underlying biological signals.


In [ ]:
%load_ext autoreload
%autoreload 2

# %% Environment Setup: Python libraries, GPU settings, and R integration
import gc
import os

import anndata2ri
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix, csc_matrix
from pipeline.utils.env import find_env_dir

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_count_matrix_location = find_env_dir("H5AD_COUNT_MATRIX")

Currently supports `.rds`, `.csv`, and `.gz`-compressed versions

In [ ]:
# Loads single file based on its format
def load_single_file(path: str) -> AnnData:
    name = path.split("/")[-1].lower()

    # Dispatcher currently supports gzipped .rds and .csv files
    if name.endswith(".rds.gz"):
        return load_rds_data(path, gz=True)
    elif name.endswith(".rds"):
        return load_rds_data(path, gz=False)
    elif name.endswith(".csv.gz") or name.endswith(".csv"):
        return load_csv_data(path)
    else:
        raise ValueError(
        f"Unsupported file type: '{name}'. "
        "This appears to be a new file type, so a new parsing logic is required."
    )

# Loads RDS.gz data
def load_rds_data(rds_path: str, gz: bool) -> AnnData:
    from rpy2.robjects import r
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    # Activate automatic conversion from R Seurat objects to Python AnnData objects
    importr("base")
    importr("Matrix")
    importr("Seurat")
    importr("SingleCellExperiment")
    
    if gz:
        read_cmd = f'readRDS(gzcon(gzfile("{rds_path}", "rb")))'
    else:
        read_cmd = f'readRDS("{rds_path}")'

    rcode = f"""
    obj <- {read_cmd}
    if (inherits(obj, "Seurat")) {{
        obj <- as.SingleCellExperiment(obj)
    }}
    assay(obj, "counts") <- as(assay(obj, "counts"), "dgCMatrix")
    # Overwrite 'X' assay with 'counts' assay
    assay(obj, "X") <- assay(obj, "counts")
    obj
    """
    with localconverter(anndata2ri.converter):
        adata = r(rcode)
        r("if (exists('obj')) rm(obj)")
        r("gc()")

    if not isinstance(adata, AnnData):
        raise ValueError("Resulting object is not an AnnData")

    adata.layers.pop("counts", None)
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    
    # Multidimensional representations (obsm, varm, layers) are intentionally cleared at this stage,
    # since integrated multi-modal / multi-dimensional analyses will be performed after dataset integration, 
    # overwriting any existing representations.
    adata.obsm = {}
    adata.varm = {}
    adata.layers = {}
    gc.collect()

    adata.obs["series"] = rds_path.split("/")[-1].split("_")[0]
    adata.obs["batch"] = "not set"
    return adata


# Loads CSV.gz data
def load_csv_data(csv_path: str) -> AnnData:
    adata: AnnData = sc.read_csv(csv_path).T

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    adata.obs["series"] = csv_path.split("/")[-1].split("_")[0]
    adata.obs["batch"] = "not set"
    return adata

# Loads 10x format data
def load_10x_data(mtx_file: str, barcodes_file: str, features_file: str, series: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    with open(barcodes_file, "r") as f:
        barcodes = [line.strip().split("\t")[0] for line in f]
    with open(features_file, "r") as f:
        genes = [line.strip().split("\t")[0] for line in f]
    adata.obs_names = barcodes
    adata.var_names = genes

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    adata.obs["series"] = series
    adata.obs["batch"] = "not set"
    return adata

def load_dense_text_matrix(matrix_file: str, annotation_file: str, series: str) -> AnnData:
    meta = pd.read_csv(annotation_file, sep="\t", index_col=0)

    CHUNK_SIZE = 4000
    sparse_chunks = []
    gene_names = []
    cell_names = None
    for i, chunk in enumerate(pd.read_csv(matrix_file, index_col=0, chunksize=CHUNK_SIZE)):
        if i == 0:
            cell_names = chunk.columns.astype(str)
        gene_names.extend(chunk.index.astype(str))
        sparse_chunks.append(csc_matrix(chunk.values))
        print(f"Processing Chunk {i}")

    X_sparse_genes_by_cells = sparse.vstack(sparse_chunks)
    X_sparse_genes_by_cells = X_sparse_genes_by_cells.transpose().tocsr()
    adata = sc.AnnData(X=X_sparse_genes_by_cells)

    adata.obs_names = cell_names # type: ignore
    adata.var_names = gene_names
    
    common_cells = adata.obs_names.intersection(meta.index)
    adata = adata[common_cells].copy()
    meta = meta.loc[common_cells]
    
    adata.obs = meta
    adata.obs["series"] = series
    adata.obs["batch"] = "not set"
    return adata

Replace `FILE` with each of your own input files and set `SAVE_FILE_NAME` accordingly
- For single files (ex. .rds, .csv, ...): Run first cell
- For 10x format (.mtx / barcodes.tsv, features.tsv): Run second cell
- For dense text matrix formats (matrix.txt / annotation.txt): Run third cell

Repeat the entire workflow below for each dataset

In [3]:
FILE = "GSE232703_final_duodenum.rds.gz"
SAVE_FILE_NAME = "GSE232703_duodenum.h5ad"
adata = load_single_file(os.path.join(raw_count_matrix_location, FILE))
adata.obs.head() #type: ignore


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,nCount_SCT,nFeature_SCT,SCT_snn_res.1,seurat_clusters,SCT_snn_res.0.5,celltype,...,broadidentity,SCT_snn_res.2,annotated,broad_identity,S.Score,G2M.Score,Phase,ident,series,batch
GGTAATCAGGCACAAC-1_1_1,vehicle_duo_female,56193,7741,0.554828,2286,892,19,37,22,epithelial5,...,EEC,37,EEC,EEC,-0.005940,-0.012746,G1,epithelial5,GSE232703,not set
CCAATGACAATTTCCT-1_1_1,vehicle_duo_female,60236,7566,1.516347,3106,839,20,34,10,gastric_surface_cell,...,paneth,34,gastric_surface_cell,gastric_surface_cell,-0.037099,-0.039733,G1,gastric_surface_cell,GSE232703,not set
AGGACGACACAGAAGC-1_1_1,vehicle_duo_female,43884,7484,1.143627,2356,1004,15,21,5,epithelial3,...,epithelial,21,epithelial3,epithelial,-0.008745,-0.025641,G1,epithelial3,GSE232703,not set
CCTCCTCGTAGCTTTG-1_1_1,vehicle_duo_female,36649,7895,0.520177,1899,487,8,19,1,enterocyte4,...,epithelial,19,enterocyte4,enterocyte,0.000000,0.000000,Undecided,enterocyte4,GSE232703,not set
ACTTATCGTTACACTG-1_1_1,vehicle_duo_female,39911,7433,0.583875,2500,877,4,23,17,+4cell,...,+4cell,23,+4cell,+4cell,0.000000,-0.007995,S,+4cell,GSE232703,not set


In [ ]:
MTX_FILE = "SCP1038_mli.matrix.mtx"
BARCODES_FILE = "SCP1038_mli.barcodes.tsv"
FEATURES_FILE = "SCP1038_mli.genes.tsv"
SERIES = "SCP1038"
SAVE_FILE_NAME = SERIES + ".h5ad"

adata = load_10x_data(
    os.path.join(raw_count_matrix_location, MTX_FILE),
    os.path.join(raw_count_matrix_location, BARCODES_FILE),
    os.path.join(raw_count_matrix_location, FEATURES_FILE),
    series = SERIES
)
adata.obs.head() #type: ignore

In [3]:
SERIES = "GSE180759"
MATRIX_FILE = "GSE180759_expression_matrix.csv.gz"
ANNOTATION_FILE = "GSE180759_annotation.txt.gz"
SAVE_FILE_NAME = SERIES + ".h5ad"

raw_count_matrix_location = os.path.join(raw_count_matrix_location, SERIES)
adata = load_dense_text_matrix(
    os.path.join(raw_count_matrix_location, MATRIX_FILE),
    os.path.join(raw_count_matrix_location, ANNOTATION_FILE),
    SERIES
)
adata.obs.head() #type: ignore

KeyboardInterrupt: 

In [ ]:
adata.obs["NBB_case"].value_counts()

/tmp/ipykernel_535775/4173232095.py:1: FutureWarning: Use obs (e.g. `k in adata.obs` or `str(adata.obs.columns.tolist())`) instead of AnnData.obs_keys, AnnData.obs_keys is deprecated and will be removed in the future.
  adata.obs_keys()


['NBB_case', 'pathology', 'seurat_cluster', 'cell_type', 'series', 'batch']

#### Reformat cell-level metadata to retain only the required information
Modify, extend, or remove parts of the code as needed, since the exact structure and annotations differ across datasets. 
In practice, you will certainly need to modify this code to accommodate the specific structure and annotations of each dataset.

In [47]:
META_DATA_FILE = "SCP1038.meta.txt"
meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")
meta.set_index("NAME", inplace=True)

meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(adata.obs, pd.DataFrame)
adata.obs = adata.obs.join(meta, how="left")

with open(os.path.join(raw_count_matrix_location, "SCP1038_mli.glia.barcodes.tsv"), "r") as f:
    enteric_glia = [line.strip().split("\t")[0] for line in f]
with open(os.path.join(raw_count_matrix_location, "SCP1038_mli.neur.barcodes.tsv"), "r") as f:
    enteric_neurons = [line.strip().split("\t")[0] for line in f]

adata.obs["celltype"] = "others"
adata.obs["celltype"].loc[adata.obs_names.isin(enteric_glia)] = "Enteric_Glia"
adata.obs["celltype"].loc[adata.obs_names.isin(enteric_neurons)] = "Enteric_Neurons"

adata.obs["sample"] = adata.obs["Unique_ID"]
adata.obs["batch"] = adata.obs["sample"]
adata.obs["doublet_batch"] = adata.obs["Overload"]
adata.obs["annotation"] = adata.obs["Annotation"]
keep = ["batch", "celltype", "sample", "series", "sex", "doublet_batch", "annotation"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

/tmp/ipykernel_233337/1343514411.py:2: DtypeWarning: Columns (1,4,5,7,9,11,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")


In [4]:
is_human = adata.obs["orig.ident"].str.contains("human", case=False)
adata = adata[~is_human].copy()

def parse_sex(s: str) -> str:
    s_low = s.lower()
    if "female" in s_low:
        return "female"
    elif "male" in s_low:
        return "male"
    return "unknown"
adata.obs["sex"] = adata.obs["orig.ident"].map(parse_sex).astype("category")

geo = adata.obs["series"].astype(str)
adata.obs["orig.ident"] = (geo + "-" + adata.obs["orig.ident"]).astype("category")
adata.obs["sample"] = adata.obs["orig.ident"]
adata.obs["batch"] = adata.obs["sample"]
adata.obs["celltype"] = adata.obs["broadidentity"]
keep = ["celltype", "sample", "series", "sex", "batch"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

Save the processed AnnData object as an `.h5ad` file and free the associated memory

In [21]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_count_matrix_location, SAVE_FILE_NAME))
del adata
gc.collect()

1372